# 🔗 Solución Ideal: Tarea 2 - Clase 7 - Normalización y Llaves

**Módulo:** Arquitectura Relacional y Bases de Datos

# Diseño Lógico: Relación 1:N (Proveedor - Producto)
 
La relación entre un **Proveedor** y un **Producto** es de **Uno a Muchos (1:N)**. 

Esto significa que un único proveedor (Ej: "Distribuidora Sabana") puede suministrar múltiples productos diferentes a nuestra cafetería (café, azúcar, vasos). Sin embargo, cada producto específico en nuestro catálogo es suministrado por un único proveedor principal.
 
**Justificación de la Llave Foránea (FK):**

Para cumplir con las reglas de **Normalización**, la tabla `productos` debe tener una Llave Foránea (`nit_proveedor`) que apunte a la Llave Primaria (`nit`) de la tabla `proveedores`. 

Si no hiciéramos esto y guardáramos el nombre y la ciudad del proveedor directamente en cada producto, estaríamos repitiendo información (Redundancia). Si el proveedor cambia de nombre o de ciudad, tendríamos que actualizar cientos de filas en la tabla de productos. Al usar una FK, garantizamos la **Integridad Referencial**: actualizamos al proveedor una sola vez en su tabla, y todos los productos vinculados reflejarán el cambio automáticamente.

In [1]:
import sqlite3

# Definimos el nombre de la base de datos
DB_NAME = "tarea_2_cafeteria_sabana_keys.db"

def inicializar_y_poblar_bd():
    
    """
    Crea las tablas normalizadas, inserta los datos de prueba y garantiza la integridad referencial.
    """
    
    # Usamos with para asegurar que la conexión se cierre automáticamente.

    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        
        # COMANDO OBLIGATORIO: Habilitar las restricciones de Llaves Foráneas en SQLite
        cursor.execute("PRAGMA foreign_keys = ON;")
        
        # Limpieza de tablas para que el script se pueda ejecutar múltiples veces sin error
        cursor.execute("DROP TABLE IF EXISTS productos;")
        cursor.execute("DROP TABLE IF EXISTS proveedores;")
        
        # ==========================================
        # 1. CREACIÓN DE TABLA: PROVEEDORES (Tabla Fuerte / Padre)
        # ==========================================
        cursor.execute('''
            CREATE TABLE proveedores (
                nit TEXT PRIMARY KEY,
                nombre_empresa TEXT NOT NULL,
                ciudad TEXT
            )
        ''')
        print("✅ Tabla 'proveedores' creada correctamente.")
        
        # ==========================================
        # 2. CREACIÓN DE TABLA: PRODUCTOS (Tabla Débil / Hijo)
        # ==========================================
        # Incluimos la Llave Foránea (FK) que apunta al NIT del proveedor
        cursor.execute('''
            CREATE TABLE productos (
                id_producto INTEGER PRIMARY KEY AUTOINCREMENT,
                nombre TEXT NOT NULL,
                precio REAL NOT NULL,
                nit_proveedor TEXT,
                FOREIGN KEY (nit_proveedor) REFERENCES proveedores(nit)
            )
        ''')
        print("✅ Tabla 'productos' creada correctamente con su Llave Foránea.")
        
        # ==========================================
        # 3. INSERCIÓN DE DATOS (Sanity Check)
        # ==========================================
        
        # Insertamos 2 Proveedores
        proveedores_data =[
            ('900111222', 'Insumos Cafeteros S.A.', 'Bogotá'),
            ('800333444', 'Distribuidora Sabana', 'Chía')
        ]

        cursor.executemany('''
            INSERT INTO proveedores (nit, nombre_empresa, ciudad) 
            VALUES (?, ?, ?)
        ''', proveedores_data)

        print(f"➕ {len(proveedores_data)} proveedores insertados.")
        
        # Insertamos 3 Productos asignando el NIT correspondiente (Validación de FK)
        # Nota: Si intentáramos poner un NIT que no existe en 'proveedores', SQLite lanzaría un IntegrityError.
        productos_data =[
            ('Café de Origen 500g', 15000.0, '900111222'),
            ('Azúcar Morena 1Kg', 4500.0, '900111222'),
            ('Vasos de Cartón x50', 8000.0, '800333444')
        ]
        
        cursor.executemany('''
            INSERT INTO productos (nombre, precio, nit_proveedor) 
            VALUES (?, ?, ?)
        ''', productos_data)

        print(f"➕ {len(productos_data)} productos insertados y vinculados a sus proveedores.")
        
        conn.commit()

# Ejecutamos la inicialización
inicializar_y_poblar_bd()

✅ Tabla 'proveedores' creada correctamente.
✅ Tabla 'productos' creada correctamente con su Llave Foránea.
➕ 2 proveedores insertados.
➕ 3 productos insertados y vinculados a sus proveedores.


# 🔍 Consulta INNER JOIN

A continuación, cruzaremos ambas tablas utilizando la Llave Foránea para reconstruir la información y mostrar un reporte consolidado y legible para el usuario.

In [2]:
def reporte_productos_proveedores():
    
    """
    Ejecuta un INNER JOIN para mostrar el nombre del producto, su precio y el nombre de la empresa proveedora.
    """

    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        
        # La consulta SQL cruza las tablas donde la FK de productos coincide con la PK de proveedores

        consulta_join = '''
            SELECT 
                productos.nombre, 
                productos.precio, 
                proveedores.nombre_empresa
            FROM productos
            INNER JOIN proveedores 
                ON productos.nit_proveedor = proveedores.nit
        '''
        
        # cursor.execute permite ejecutar la consulta y fetchall() recupera todos los resultados en una lista de tuplas.
        cursor.execute(consulta_join)
        resultados = cursor.fetchall()
        
        # Impresión formateada del reporte
        print("\n" + "="*60)
        print("📊 REPORTE DE INVENTARIO Y PROVEEDORES (INNER JOIN)")
        print("="*60)
        
        for fila in resultados:
            nombre_prod = fila[0]
            precio_prod = fila[1]
            nombre_prov = fila[2]
            
            # Usamos ljust() para alinear el texto y f-strings para el formato de moneda
            print(f"🛒 Producto: {nombre_prod.ljust(22)} | Precio: ${precio_prod:,.2f} | 🏢 Proveedor: {nombre_prov}")
            
        print("="*60)

# Ejecutamos el reporte
reporte_productos_proveedores()


📊 REPORTE DE INVENTARIO Y PROVEEDORES (INNER JOIN)
🛒 Producto: Café de Origen 500g    | Precio: $15,000.00 | 🏢 Proveedor: Insumos Cafeteros S.A.
🛒 Producto: Azúcar Morena 1Kg      | Precio: $4,500.00 | 🏢 Proveedor: Insumos Cafeteros S.A.
🛒 Producto: Vasos de Cartón x50    | Precio: $8,000.00 | 🏢 Proveedor: Distribuidora Sabana


# 🔍 Insertar Nuevo Producto con Input

In [3]:
def insertar_nuevo_producto():
    
    """
    Permite insertar un nuevo producto interactivamente solicitando datos al usuario.
    """
    
    with sqlite3.connect(DB_NAME) as conn:
        cursor = conn.cursor()
        cursor.execute("PRAGMA foreign_keys = ON;")
        
        print("\n" + "="*60)
        print("➕ INSERTAR NUEVO PRODUCTO")
        print("="*60)
        
        # Primero, mostramos los proveedores disponibles
        cursor.execute("SELECT nit, nombre_empresa FROM proveedores")
        proveedores = cursor.fetchall()
        
        print("\n📋 Proveedores disponibles:")
        for i, (nit, nombre) in enumerate(proveedores, 1):
            print(f"  {i}. {nombre} (NIT: {nit})")
        
        # Solicitamos datos del usuario
        nombre_producto = input("\n🏷️  Ingrese el nombre del producto: ").strip()
        
        try:
            precio_producto = float(input("💰 Ingrese el precio del producto: "))
        except ValueError:
            print("❌ Error: El precio debe ser un número válido.")
            return
        
        opcion_proveedor = input(f"🏢 Seleccione el número del proveedor (1-{len(proveedores)}): ").strip()
        
        try:
            opcion_idx = int(opcion_proveedor) - 1
            if 0 <= opcion_idx < len(proveedores):
                nit_seleccionado = proveedores[opcion_idx][0]
            else:
                print("❌ Error: Opción de proveedor inválida.")
                return
        except ValueError:
            print("❌ Error: Ingrese un número válido.")
            return
        
        # Insertamos el nuevo producto
        try:
            cursor.execute('''
                INSERT INTO productos (nombre, precio, nit_proveedor) 
                VALUES (?, ?, ?)
            ''', (nombre_producto, precio_producto, nit_seleccionado))
            
            conn.commit()
            print(f"\n✅ Producto '{nombre_producto}' insertado correctamente.")
            
        except sqlite3.IntegrityError as e:
            print(f"❌ Error de integridad: {e}")

# Ejecutamos la función para insertar un nuevo producto
insertar_nuevo_producto()

# Ejecutamos el reporte actualizado
print("\n")
reporte_productos_proveedores()


➕ INSERTAR NUEVO PRODUCTO

📋 Proveedores disponibles:
  1. Insumos Cafeteros S.A. (NIT: 900111222)
  2. Distribuidora Sabana (NIT: 800333444)

✅ Producto 'Arepas' insertado correctamente.



📊 REPORTE DE INVENTARIO Y PROVEEDORES (INNER JOIN)
🛒 Producto: Café de Origen 500g    | Precio: $15,000.00 | 🏢 Proveedor: Insumos Cafeteros S.A.
🛒 Producto: Azúcar Morena 1Kg      | Precio: $4,500.00 | 🏢 Proveedor: Insumos Cafeteros S.A.
🛒 Producto: Vasos de Cartón x50    | Precio: $8,000.00 | 🏢 Proveedor: Distribuidora Sabana
🛒 Producto: Arepas                 | Precio: $2,000.00 | 🏢 Proveedor: Distribuidora Sabana
